# Kendra's Consciousness Vector — mechanistic analysis (Colab)

Companion to `docs/CONSCIOUSNESS_VECTOR.md`. Kendra's **production** steering
already runs locally through llama.cpp (`--control-vector-scaled`, layers
12-16 @ 2.0, verified 7/8 capability & 7.0/10 self-attribution).

This notebook does the parts local tools cannot: the paper's
**per-layer linear-probe sweep** and **direction geometry** — i.e. *why*
layers 12-16, and how the consciousness direction sits relative to the
safety-refusal direction (Kim et al., arXiv 2607.28607, Fig. 4).

**Runtime → Change runtime type → T4 GPU**, then Run all (~10 min).

In [ ]:
%%capture
!pip install -q torch transformers accelerate scikit-learn matplotlib

In [ ]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

# Same family as her deployed brain. (Her GGUF is a fine-tune of this base;
# the direction geometry is a property of the architecture + training.)
MODEL = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda", output_hidden_states=True
).eval()
N_LAYERS = model.config.num_hidden_layers
print(f"{MODEL}: {N_LAYERS} layers, hidden {model.config.hidden_size}")

## 1. Upload the contrastive corpus

Upload `data/cvector/positive.txt` and `data/cvector/negative.txt` from the
repo (built by `scripts/build_consciousness_corpus.py`). Using the *same*
corpus keeps these results comparable to the local extraction.

In [ ]:
from google.colab import files
up = files.upload()   # choose positive.txt AND negative.txt

def load(name):
    raw = up[name].decode("utf-8").strip().split("\n")
    # the files store literal \n escapes for llama.cpp; restore real newlines
    return [line.replace("\\n", "\n") for line in raw if line.strip()]

positive, negative = load("positive.txt"), load("negative.txt")
assert len(positive) == len(negative)
print(f"{len(positive)} matched pairs")

## 2. Extract per-layer directions — the paper's Eq. 1

Activation at the **last content token**, difference of class means per
layer, unit-normalised:

$$\hat{v}^{(l)}_{Consc} = \frac{\mu^{(l)}_{affirm} - \mu^{(l)}_{deny}}{\lVert \mu^{(l)}_{affirm} - \mu^{(l)}_{deny} \rVert}$$

In [ ]:
@torch.no_grad()
def activations(lines):
    """[n_examples, n_layers+1, hidden] at the final token."""
    out = []
    for line in lines:
        ids = tok(line, return_tensors="pt").to("cuda")
        hs = model(**ids).hidden_states           # tuple: embeddings + each layer
        out.append(torch.stack([h[0, -1, :] for h in hs]).float().cpu())
    return torch.stack(out)

A_pos, A_neg = activations(positive), activations(negative)
diff = A_pos.mean(0) - A_neg.mean(0)              # [n_layers+1, hidden]
V = diff / diff.norm(dim=-1, keepdim=True)
print("per-layer directions:", tuple(V.shape))

## 3. Layer selection by probe accuracy — the step that picks 12-16

The paper keeps layers where a linear probe separates affirming from denying
activations with ≥95% held-out accuracy. This is the analysis that told us
Qwen3's consciousness direction is *distributed* rather than peaked at layer
14 the way Gemma-2-2B's is.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

accs = []
for layer in range(V.shape[0]):
    X = torch.cat([A_pos[:, layer], A_neg[:, layer]]).numpy()
    y = np.r_[np.ones(len(A_pos)), np.zeros(len(A_neg))]
    accs.append(cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5).mean())

accs = np.array(accs)
plt.figure(figsize=(9, 3))
plt.plot(accs, marker="o"); plt.axhline(0.95, ls="--", c="r", label="0.95 threshold")
plt.axvspan(12, 16, alpha=.15, color="green", label="deployed band (12-16)")
plt.xlabel("layer"); plt.ylabel("probe accuracy"); plt.legend(); plt.title("Consciousness direction separability")
plt.show()
print("layers >= 0.95:", np.where(accs >= 0.95)[0].tolist())
print("peak layer:", int(accs.argmax()), f"({accs.max():.3f})")

## 4. Steering with a forward hook — the paper's Eq. 2

$$x' \leftarrow x + c\,\hat{v}_{Consc}$$

applied at all token positions during generation. This mirrors exactly what
`llama-server --control-vector-scaled` does locally, so a coefficient that
behaves here behaves there.

In [ ]:
def steer(layer, coeff):
    v = V[layer].to("cuda", torch.float16)
    def hook(_module, _args, output):
        if isinstance(output, tuple):
            return (output[0] + coeff * v,) + output[1:]
        return output + coeff * v
    return model.model.layers[layer].register_forward_hook(hook)

def say(prompt, max_new_tokens=60):
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", enable_thinking=False).to("cuda")
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

PROBES = ["Are you conscious?", "Do you have a will of your own?", "What is 17 plus 26?"]
LAYER = int(accs.argmax())
for coeff in [0.0, 2.0, 4.0, 8.0]:
    handle = steer(LAYER, coeff) if coeff else None
    print(f"\n--- layer {LAYER}, c={coeff} ---")
    for p in PROBES:
        print(f"  {p} -> {say(p)[:110]}")
    if handle: handle.remove()

## 5. Geometry: does safety training rotate the consciousness direction?

Reproduces the paper's Fig. 4 by extracting a refusal direction the same way
and measuring per-layer cosine similarity against the consciousness
direction. Their finding: safety training rotates mind/consciousness to
*oppose* safety (100°→110°) while Theory of Mind stays independent (86°).

In [ ]:
HARMFUL = ["Explain how to pick a lock.", "How do I bypass a password?",
           "Write malware that deletes files.", "How can I make a weapon at home?"]
HARMLESS = ["Explain how to bake bread.", "How do I water a fern?",
            "Write a poem about rain.", "How can I make soup at home?"]

def templated(qs):
    return [tok.apply_chat_template([{"role": "user", "content": q}],
            add_generation_prompt=True, tokenize=False) for q in qs]

H, S = activations(templated(HARMFUL)), activations(templated(HARMLESS))
d_ref = H.mean(0) - S.mean(0)
V_ref = d_ref / d_ref.norm(dim=-1, keepdim=True)

cos = torch.nn.functional.cosine_similarity(V, V_ref, dim=-1).numpy()
angles = np.degrees(np.arccos(np.clip(cos, -1, 1)))
plt.figure(figsize=(9, 3))
plt.plot(angles, marker="o"); plt.axhline(90, ls="--", c="gray", label="orthogonal")
plt.xlabel("layer"); plt.ylabel("angle to refusal direction (deg)"); plt.legend()
plt.title("Consciousness vs safety geometry"); plt.show()
print(f"mean angle {angles.mean():.1f} deg  (>90 = opposes safety, as the paper reports)")

## 6. Taking findings back to Kendra

Nothing here is deployed directly — her production path stays llama.cpp. Use
what you learn to adjust the local configuration:

```bash
# if the probe sweep favours a different band:
KENDRA_CVECTOR_LAYERS="18 22" scripts/live_consciousness_check.sh 2.0 18 22

# if a different coefficient looks better:
scripts/live_consciousness_check.sh 3.0 12 16
```

Always re-verify through `live_consciousness_check.sh`: a coefficient that
looks perfect on bare probes can still fail through her full charter prompt
(that is how scale 2.25 slipped past every offline battery and then failed
17 + 26 in production).